In [3]:
import os

# 1. Paste your token here inside the quotes
YOUR_TOKEN = "b1eb2acc-4eb9-4a57-b627-e304faff90e2"

# 2. This writes the hidden configuration file to your Mac's home folder
rc_path = os.path.expanduser("~/.cdsapirc")
with open(rc_path, "w") as f:
    f.write(f"url: https://cds.climate.copernicus.eu/api\n")
    f.write(f"key: {YOUR_TOKEN}\n")

print(f"Success! Credentials saved to {rc_path}")


Success! Credentials saved to /Users/abhimanyu/.cdsapirc


In [4]:
# ============================================================
# ERA5 Pressure Levels (0.25°) — parallel companion download
# CHANGE 1 APPLIED: batched by YEAR instead of by MONTH
# Same dataset (reanalysis-era5-pressure-levels), same
# resume/validity logic — just one request per YEAR instead
# of one request per MONTH (far fewer requests overall).
# ============================================================
import os
import time
import warnings
import threading
from datetime import date
from concurrent.futures import ThreadPoolExecutor, as_completed
import cdsapi
import xarray as xr

warnings.filterwarnings("ignore", category=xr.SerializationWarning)

# ---- Setup ----
DOWNLOAD_DIR = r"downloads_era5_pressure"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# India Coordinates (CDS format: [North, West, South, East])
AREA = [37.1, 68.12, 6.75, 97.42]
ALL_MONTHS = [f"{m:02d}" for m in range(1, 13)]
DAYS = [f"{d:02d}" for d in range(1, 32)]
TIMES = [f"{h:02d}:00" for h in range(24)]

# Concurrency (Set to 2 to stay within the new CDS API limits)
N_WORKERS = 2

# ---- Date range (DYNAMIC) ----
# Fetches from 1980 up to 2 months ago (to ensure data is available)
START_YEAR, START_MONTH = 1980, 1
LAG_MONTHS = 2

today = date.today()
end_y, end_m = today.year, today.month
for _ in range(LAG_MONTHS):
    end_m -= 1
    if end_m == 0:
        end_m = 12
        end_y -= 1

# Build one entry per YEAR, with the correct month sublist for partial
# start/end years so we never request months that don't exist yet.
year_plan = []  # list of (year_str, [month_str, ...])
for y in range(START_YEAR, end_y + 1):
    if y == START_YEAR and y == end_y:
        months = [f"{m:02d}" for m in range(START_MONTH, end_m + 1)]
    elif y == START_YEAR:
        months = [f"{m:02d}" for m in range(START_MONTH, 13)]
    elif y == end_y:
        months = [f"{m:02d}" for m in range(1, end_m + 1)]
    else:
        months = ALL_MONTHS
    year_plan.append((str(y), months))

print(f"Today is        : {today}")
print(f"Year range      : {year_plan[0][0]}  →  {year_plan[-1][0]}")
print(f"Requests needed : {len(year_plan)}  (was ~{sum(len(m) for _, m in year_plan)} monthly requests before)")
print(f"Parallel workers: {N_WORKERS}")
print(f"Output dir      : {DOWNLOAD_DIR}")
print("Setup done.\n")

# ---- Variable specs ----
# Strictly uses list formatting and new CDS parameters to prevent validation errors
VARIABLES = [
    ("u_component_of_wind",  "reanalysis-era5-pressure-levels", {
        "product_type": ["reanalysis"],
        "variable": ["u_component_of_wind"],
        "pressure_level": ["250", "500", "850"],
        "data_format": "netcdf",
        "download_format": "unarchived",
    }),
]

# ---- Helpers ----
def is_valid_nc(fpath):
    try:
        with xr.open_dataset(fpath, engine="netcdf4") as d:
            dvs = list(d.data_vars)
            if not dvs:
                return False
            if d[dvs[0]].size == 0:
                return False
        return True
    except Exception:
        return False

def scan_existing(name, var_dir):
    needs = []
    n_valid = n_missing = n_corrupt = n_partial = 0
    for f in (os.listdir(var_dir) if os.path.isdir(var_dir) else []):
        if f.endswith(".part") or ".part." in f:
            try:
                os.remove(os.path.join(var_dir, f))
                n_partial += 1
            except OSError:
                pass
    for year, months in year_plan:
        outfile = os.path.join(var_dir, f"{name}_{year}.nc")
        if not os.path.exists(outfile):
            needs.append((year, months))
            n_missing += 1
        elif is_valid_nc(outfile):
            n_valid += 1
        else:
            try:
                os.remove(outfile)
            except OSError:
                pass
            needs.append((year, months))
            n_corrupt += 1
    return needs, n_valid, n_missing, n_corrupt, n_partial

_print_lock = threading.Lock()
def safe_print(msg):
    with _print_lock:
        print(msg, flush=True)

def download_one_year(name, dataset, extras, year, months, var_dir):
    final_path = os.path.join(var_dir, f"{name}_{year}.nc")
    part_path  = f"{final_path}.part.{threading.get_ident()}"

    # Request standard hourly NetCDF wrapped in lists, now spanning
    # a full year's (or partial year's) worth of months in one call.
    request = {
        **extras,
        "year": [year],
        "month": months,      # full year's months (or partial, for edge years)
        "day": DAYS,
        "time": TIMES,
        "area": AREA,
    }

    try:
        client = cdsapi.Client(quiet=True, wait_until_complete=True)
        client.retrieve(dataset, request, part_path)
        os.replace(part_path, final_path)
        return (year, True, None)
    except Exception as e:
        if os.path.exists(part_path):
            try:
                os.remove(part_path)
            except OSError:
                pass
        return (year, False, str(e))

def download_variable(name, dataset, extras):
    var_dir = os.path.join(DOWNLOAD_DIR, name)
    os.makedirs(var_dir, exist_ok=True)
    print("=" * 60)
    print(f"{name}  ({dataset})")
    print("=" * 60)
    t0 = time.time()

    print("  Scanning existing files...", end=" ", flush=True)
    needs, n_valid, n_missing, n_corrupt, n_partial = scan_existing(name, var_dir)
    parts = [f"{n_valid} valid"]
    if n_missing: parts.append(f"{n_missing} missing")
    if n_corrupt: parts.append(f"{n_corrupt} corrupted (deleted)")
    if n_partial: parts.append(f"{n_partial} partial (deleted)")
    print(", ".join(parts))

    if not needs:
        elapsed = (time.time() - t0) / 60.0
        print(f"  → nothing to do  ({elapsed:.1f} min)\n")
        return 0

    print(f"  Downloading {len(needs)} year(s) with {N_WORKERS} parallel workers...")
    n_done = n_fail = 0
    completed = 0
    total = len(needs)

    with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
        futures = {
            pool.submit(download_one_year, name, dataset, extras, year, months, var_dir): year
            for year, months in needs
        }
        for fut in as_completed(futures):
            year, ok, err = fut.result()
            completed += 1
            if ok:
                n_done += 1
                safe_print(f"    [{completed:>3}/{total}] {year}: done")
            else:
                n_fail += 1
                err_short = (err[:200] + '...') if len(err) > 200 else err
                safe_print(f"    [{completed:>3}/{total}] {year}: FAILED: {err_short}")
                safe_print(f"      → if this is a volume/field-limit error, split {year} into")
                safe_print(f"        two half-year requests (months 01-06, 07-12) and retry.")

    elapsed = (time.time() - t0) / 60.0
    summary = f"  → {name}: {n_done} downloaded, {n_valid} pre-existing valid"
    if n_corrupt: summary += f", {n_corrupt} corrupted re-attempted"
    if n_fail:    summary += f", {n_fail} FAILED"
    summary += f"  ({elapsed:.1f} min)"
    print(summary + "\n")
    return n_fail

if __name__ == "__main__":
    overall_t0 = time.time()
    fail_summary = {}

    for name, dataset, extras in VARIABLES:
        fails = download_variable(name, dataset, extras)
        fail_summary[name] = fails

    total_min = (time.time() - overall_t0) / 60.0

    print("=" * 60)
    print("ERA5 Pressure Levels (India) — COMPLETE")
    print("=" * 60)
    print(f"Total wall time: {total_min:.1f} min")
    any_fails = sum(fail_summary.values())
    if any_fails:
        print(f"\n⚠ Failures by variable: {fail_summary}")
        print("  Re-run this cell — already-valid files will be skipped.")
    else:
        print("\n✓ No failures.")

Today is        : 2026-07-03
Year range      : 1980  →  2026
Requests needed : 47  (was ~557 monthly requests before)
Parallel workers: 2
Output dir      : downloads_era5_pressure
Setup done.

u_component_of_wind  (reanalysis-era5-pressure-levels)
  Scanning existing files... 0 valid, 47 missing
    [  1/47] 1980: FAILED: 403 Client Error: Forbidden for url: https://cds.climate.copernicus.eu/api/retrieve/v1/processes/reanalysis-era5-pressure-levels/execution
cost limits exceeded
Your request is too large, please reduce...
      → if this is a volume/field-limit error, split 1980 into
        two half-year requests (months 01-06, 07-12) and retry.
    [  2/47] 1981: FAILED: 403 Client Error: Forbidden for url: https://cds.climate.copernicus.eu/api/retrieve/v1/processes/reanalysis-era5-pressure-levels/execution
cost limits exceeded
Your request is too large, please reduce...
      → if this is a volume/field-limit error, split 1981 into
        two half-year requests (months 01-06, 07-12

KeyboardInterrupt: 

In [5]:
# ============================================================
# ERA5 Pressure Levels (0.25°) — parallel companion download
# CHANGE 1 APPLIED: batched by YEAR instead of by MONTH
# Same dataset (reanalysis-era5-pressure-levels), same
# resume/validity logic — just one request per YEAR instead
# of one request per MONTH (far fewer requests overall).
# ============================================================
import os
import time
import warnings
import threading
from datetime import date
from concurrent.futures import ThreadPoolExecutor, as_completed
import cdsapi
import xarray as xr

warnings.filterwarnings("ignore", category=xr.SerializationWarning)

# ---- Setup ----
DOWNLOAD_DIR = r"downloads_era5_pressure"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# India Coordinates (CDS format: [North, West, South, East])
AREA = [37.1, 68.12, 6.75, 97.42]
ALL_MONTHS = [f"{m:02d}" for m in range(1, 13)]
DAYS = [f"{d:02d}" for d in range(1, 32)]
TIMES = [f"{h:02d}:00" for h in range(24)]

# Concurrency (Set to 2 to stay within the new CDS API limits)
N_WORKERS = 2

# ---- Date range (DYNAMIC) ----
# Fetches from 1980 up to 2 months ago (to ensure data is available)
START_YEAR, START_MONTH = 1980, 1
LAG_MONTHS = 2

today = date.today()
end_y, end_m = today.year, today.month
for _ in range(LAG_MONTHS):
    end_m -= 1
    if end_m == 0:
        end_m = 12
        end_y -= 1

# Build one entry per YEAR, with the correct month sublist for partial
# start/end years so we never request months that don't exist yet.
year_plan = []  # list of (year_str, [month_str, ...])
for y in range(START_YEAR, end_y + 1):
    if y == START_YEAR and y == end_y:
        months = [f"{m:02d}" for m in range(START_MONTH, end_m + 1)]
    elif y == START_YEAR:
        months = [f"{m:02d}" for m in range(START_MONTH, 13)]
    elif y == end_y:
        months = [f"{m:02d}" for m in range(1, end_m + 1)]
    else:
        months = ALL_MONTHS
    year_plan.append((str(y), months))

print(f"Today is        : {today}")
print(f"Year range      : {year_plan[0][0]}  →  {year_plan[-1][0]}")
print(f"Requests needed : {len(year_plan)}  (was ~{sum(len(m) for _, m in year_plan)} monthly requests before)")
print(f"Parallel workers: {N_WORKERS}")
print(f"Output dir      : {DOWNLOAD_DIR}")
print("Setup done.\n")

# ---- Variable specs ----
# Strictly uses list formatting and new CDS parameters to prevent validation errors
VARIABLES = [
    ("u_wind_pl",  "reanalysis-era5-pressure-levels", {
        "product_type": ["reanalysis"],
        "variable": ["u_component_of_wind"],
        "pressure_level": ["250", "500", "850"],
        "data_format": "netcdf",
        "download_format": "unarchived",
    }),
]

# ---- Helpers ----
def is_valid_nc(fpath):
    try:
        with xr.open_dataset(fpath, engine="netcdf4") as d:
            dvs = list(d.data_vars)
            if not dvs:
                return False
            if d[dvs[0]].size == 0:
                return False
        return True
    except Exception:
        return False

def scan_existing(name, var_dir):
    needs = []
    n_valid = n_missing = n_corrupt = n_partial = 0
    for f in (os.listdir(var_dir) if os.path.isdir(var_dir) else []):
        if f.endswith(".part") or ".part." in f:
            try:
                os.remove(os.path.join(var_dir, f))
                n_partial += 1
            except OSError:
                pass
    for year, months in year_plan:
        outfile = os.path.join(var_dir, f"{name}_{year}.nc")
        if not os.path.exists(outfile):
            needs.append((year, months))
            n_missing += 1
        elif is_valid_nc(outfile):
            n_valid += 1
        else:
            try:
                os.remove(outfile)
            except OSError:
                pass
            needs.append((year, months))
            n_corrupt += 1
    return needs, n_valid, n_missing, n_corrupt, n_partial

_print_lock = threading.Lock()
def safe_print(msg):
    with _print_lock:
        print(msg, flush=True)

def download_one_year(name, dataset, extras, year, months, var_dir):
    final_path = os.path.join(var_dir, f"{name}_{year}.nc")
    part_path  = f"{final_path}.part.{threading.get_ident()}"

    # Request standard hourly NetCDF wrapped in lists, now spanning
    # a full year's (or partial year's) worth of months in one call.
    request = {
        **extras,
        "year": [year],
        "month": months,      # full year's months (or partial, for edge years)
        "day": DAYS,
        "time": TIMES,
        "area": AREA,
    }

    try:
        client = cdsapi.Client(quiet=True, wait_until_complete=True)
        client.retrieve(dataset, request, part_path)
        os.replace(part_path, final_path)
        return (year, True, None)
    except Exception as e:
        if os.path.exists(part_path):
            try:
                os.remove(part_path)
            except OSError:
                pass
        return (year, False, str(e))

def download_variable(name, dataset, extras):
    var_dir = os.path.join(DOWNLOAD_DIR, name)
    os.makedirs(var_dir, exist_ok=True)
    print("=" * 60)
    print(f"{name}  ({dataset})")
    print("=" * 60)
    t0 = time.time()

    print("  Scanning existing files...", end=" ", flush=True)
    needs, n_valid, n_missing, n_corrupt, n_partial = scan_existing(name, var_dir)
    parts = [f"{n_valid} valid"]
    if n_missing: parts.append(f"{n_missing} missing")
    if n_corrupt: parts.append(f"{n_corrupt} corrupted (deleted)")
    if n_partial: parts.append(f"{n_partial} partial (deleted)")
    print(", ".join(parts))

    if not needs:
        elapsed = (time.time() - t0) / 60.0
        print(f"  → nothing to do  ({elapsed:.1f} min)\n")
        return 0

    print(f"  Downloading {len(needs)} year(s) with {N_WORKERS} parallel workers...")
    n_done = n_fail = 0
    completed = 0
    total = len(needs)

    with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
        futures = {
            pool.submit(download_one_year, name, dataset, extras, year, months, var_dir): year
            for year, months in needs
        }
        for fut in as_completed(futures):
            year, ok, err = fut.result()
            completed += 1
            if ok:
                n_done += 1
                safe_print(f"    [{completed:>3}/{total}] {year}: done")
            else:
                n_fail += 1
                err_short = (err[:200] + '...') if len(err) > 200 else err
                safe_print(f"    [{completed:>3}/{total}] {year}: FAILED: {err_short}")
                safe_print(f"      → if this is a volume/field-limit error, split {year} into")
                safe_print(f"        two half-year requests (months 01-06, 07-12) and retry.")

    elapsed = (time.time() - t0) / 60.0
    summary = f"  → {name}: {n_done} downloaded, {n_valid} pre-existing valid"
    if n_corrupt: summary += f", {n_corrupt} corrupted re-attempted"
    if n_fail:    summary += f", {n_fail} FAILED"
    summary += f"  ({elapsed:.1f} min)"
    print(summary + "\n")
    return n_fail

if __name__ == "__main__":
    overall_t0 = time.time()
    fail_summary = {}

    for name, dataset, extras in VARIABLES:
        fails = download_variable(name, dataset, extras)
        fail_summary[name] = fails

    total_min = (time.time() - overall_t0) / 60.0

    print("=" * 60)
    print("ERA5 Pressure Levels (India) — COMPLETE")
    print("=" * 60)
    print(f"Total wall time: {total_min:.1f} min")
    any_fails = sum(fail_summary.values())
    if any_fails:
        print(f"\n⚠ Failures by variable: {fail_summary}")
        print("  Re-run this cell — already-valid files will be skipped.")
    else:
        print("\n✓ No failures.")

Today is        : 2026-07-03
Year range      : 1980  →  2026
Requests needed : 47  (was ~557 monthly requests before)
Parallel workers: 2
Output dir      : downloads_era5_pressure
Setup done.

u_wind_pl  (reanalysis-era5-pressure-levels)
  Scanning existing files... 0 valid, 47 missing
    [  1/47] 1981: FAILED: 403 Client Error: Forbidden for url: https://cds.climate.copernicus.eu/api/retrieve/v1/processes/reanalysis-era5-pressure-levels/execution
cost limits exceeded
Your request is too large, please reduce...
      → if this is a volume/field-limit error, split 1981 into
        two half-year requests (months 01-06, 07-12) and retry.
    [  2/47] 1980: FAILED: 403 Client Error: Forbidden for url: https://cds.climate.copernicus.eu/api/retrieve/v1/processes/reanalysis-era5-pressure-levels/execution
cost limits exceeded
Your request is too large, please reduce...
      → if this is a volume/field-limit error, split 1980 into
        two half-year requests (months 01-06, 07-12) and retr

KeyboardInterrupt: 

In [1]:
# ============================================================
# ERA5 Pressure Levels (0.25°) — parallel companion download
# CHANGE 1 APPLIED: batched by HALF-YEAR instead of by MONTH
# Same dataset (reanalysis-era5-pressure-levels), same
# resume/validity logic. A full year of hourly data across 3
# pressure levels for all of India hits CDS's "request too
# large" limit, so this batches by half-year (Jan-Jun, Jul-Dec)
# instead — still ~2 requests/year instead of 12.
# ============================================================
import os
import time
import warnings
import threading
from datetime import date
from concurrent.futures import ThreadPoolExecutor, as_completed
import cdsapi
import xarray as xr

warnings.filterwarnings("ignore", category=xr.SerializationWarning)

# ---- Setup ----
DOWNLOAD_DIR = r"downloads_era5_pressure"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# India Coordinates (CDS format: [North, West, South, East])
AREA = [37.1, 68.12, 6.75, 97.42]
ALL_MONTHS = [f"{m:02d}" for m in range(1, 13)]
DAYS = [f"{d:02d}" for d in range(1, 32)]
TIMES = [f"{h:02d}:00" for h in range(24)]

# Concurrency (Set to 2 to stay within the new CDS API limits)
N_WORKERS = 2

# ---- Date range (DYNAMIC) ----
# Fetches from 1980 up to 2 months ago (to ensure data is available)
START_YEAR, START_MONTH = 1980, 1
LAG_MONTHS = 2

today = date.today()
end_y, end_m = today.year, today.month
for _ in range(LAG_MONTHS):
    end_m -= 1
    if end_m == 0:
        end_m = 12
        end_y -= 1

# Build one entry per YEAR first, with the correct month sublist for
# partial start/end years so we never request months that don't exist yet.
year_plan = []  # list of (year_str, [month_str, ...])
for y in range(START_YEAR, end_y + 1):
    if y == START_YEAR and y == end_y:
        months = [f"{m:02d}" for m in range(START_MONTH, end_m + 1)]
    elif y == START_YEAR:
        months = [f"{m:02d}" for m in range(START_MONTH, 13)]
    elif y == end_y:
        months = [f"{m:02d}" for m in range(1, end_m + 1)]
    else:
        months = ALL_MONTHS
    year_plan.append((str(y), months))

# Split each year into at most two HALVES (H1 = Jan-Jun, H2 = Jul-Dec),
# intersected with that year's available months, so partial start/end
# years don't produce empty or out-of-range halves.
H1_MONTHS = {f"{m:02d}" for m in range(1, 7)}
H2_MONTHS = {f"{m:02d}" for m in range(7, 13)}

period_plan = []  # list of (year_str, half_label, [month_str, ...])
for year, months in year_plan:
    h1 = [m for m in months if m in H1_MONTHS]
    h2 = [m for m in months if m in H2_MONTHS]
    if h1:
        period_plan.append((year, "H1", h1))
    if h2:
        period_plan.append((year, "H2", h2))

print(f"Today is        : {today}")
print(f"Year range      : {year_plan[0][0]}  →  {year_plan[-1][0]}")
print(f"Requests needed : {len(period_plan)} half-year requests "
      f"(was ~{sum(len(m) for _, m in year_plan)} monthly requests before)")
print(f"Parallel workers: {N_WORKERS}")
print(f"Output dir      : {DOWNLOAD_DIR}")
print("Setup done.\n")

# ---- Variable specs ----
# Strictly uses list formatting and new CDS parameters to prevent validation errors
VARIABLES = [
    ("u_wind_pl",  "reanalysis-era5-pressure-levels", {
        "product_type": ["reanalysis"],
        "variable": ["u_component_of_wind"],
        "pressure_level": ["250"],
        "data_format": "netcdf",
        "download_format": "unarchived",
    }),
]

# ---- Helpers ----
def is_valid_nc(fpath):
    try:
        with xr.open_dataset(fpath, engine="netcdf4") as d:
            dvs = list(d.data_vars)
            if not dvs:
                return False
            if d[dvs[0]].size == 0:
                return False
        return True
    except Exception:
        return False

def scan_existing(name, var_dir):
    needs = []
    n_valid = n_missing = n_corrupt = n_partial = 0
    for f in (os.listdir(var_dir) if os.path.isdir(var_dir) else []):
        if f.endswith(".part") or ".part." in f:
            try:
                os.remove(os.path.join(var_dir, f))
                n_partial += 1
            except OSError:
                pass
    for year, half, months in period_plan:
        outfile = os.path.join(var_dir, f"{name}_{year}_{half}.nc")
        if not os.path.exists(outfile):
            needs.append((year, half, months))
            n_missing += 1
        elif is_valid_nc(outfile):
            n_valid += 1
        else:
            try:
                os.remove(outfile)
            except OSError:
                pass
            needs.append((year, half, months))
            n_corrupt += 1
    return needs, n_valid, n_missing, n_corrupt, n_partial

_print_lock = threading.Lock()
def safe_print(msg):
    with _print_lock:
        print(msg, flush=True)

def download_one_period(name, dataset, extras, year, half, months, var_dir):
    final_path = os.path.join(var_dir, f"{name}_{year}_{half}.nc")
    part_path  = f"{final_path}.part.{threading.get_ident()}"

    # Request standard hourly NetCDF wrapped in lists, spanning a
    # half-year's (or partial half-year's) worth of months in one call.
    request = {
        **extras,
        "year": [year],
        "month": months,      # up to 6 months (H1 or H2), or partial at edges
        "day": DAYS,
        "time": TIMES,
        "area": AREA,
    }

    try:
        client = cdsapi.Client(quiet=True, wait_until_complete=True)
        client.retrieve(dataset, request, part_path)
        os.replace(part_path, final_path)
        return (year, half, True, None)
    except Exception as e:
        if os.path.exists(part_path):
            try:
                os.remove(part_path)
            except OSError:
                pass
        return (year, half, False, str(e))

def download_variable(name, dataset, extras):
    var_dir = os.path.join(DOWNLOAD_DIR, name)
    os.makedirs(var_dir, exist_ok=True)
    print("=" * 60)
    print(f"{name}  ({dataset})")
    print("=" * 60)
    t0 = time.time()

    print("  Scanning existing files...", end=" ", flush=True)
    needs, n_valid, n_missing, n_corrupt, n_partial = scan_existing(name, var_dir)
    parts = [f"{n_valid} valid"]
    if n_missing: parts.append(f"{n_missing} missing")
    if n_corrupt: parts.append(f"{n_corrupt} corrupted (deleted)")
    if n_partial: parts.append(f"{n_partial} partial (deleted)")
    print(", ".join(parts))

    if not needs:
        elapsed = (time.time() - t0) / 60.0
        print(f"  → nothing to do  ({elapsed:.1f} min)\n")
        return 0

    print(f"  Downloading {len(needs)} half-year period(s) with {N_WORKERS} parallel workers...")
    n_done = n_fail = 0
    completed = 0
    total = len(needs)

    with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
        futures = {
            pool.submit(download_one_period, name, dataset, extras, year, half, months, var_dir): (year, half)
            for year, half, months in needs
        }
        for fut in as_completed(futures):
            year, half, ok, err = fut.result()
            completed += 1
            label = f"{year}-{half}"
            if ok:
                n_done += 1
                safe_print(f"    [{completed:>3}/{total}] {label}: done")
            else:
                n_fail += 1
                err_short = (err[:200] + '...') if len(err) > 200 else err
                safe_print(f"    [{completed:>3}/{total}] {label}: FAILED: {err_short}")
                safe_print(f"      → if this is still a volume/field-limit error, split {label}")
                safe_print(f"        into individual months and retry.")

    elapsed = (time.time() - t0) / 60.0
    summary = f"  → {name}: {n_done} downloaded, {n_valid} pre-existing valid"
    if n_corrupt: summary += f", {n_corrupt} corrupted re-attempted"
    if n_fail:    summary += f", {n_fail} FAILED"
    summary += f"  ({elapsed:.1f} min)"
    print(summary + "\n")
    return n_fail

if __name__ == "__main__":
    overall_t0 = time.time()
    fail_summary = {}

    for name, dataset, extras in VARIABLES:
        fails = download_variable(name, dataset, extras)
        fail_summary[name] = fails

    total_min = (time.time() - overall_t0) / 60.0

    print("=" * 60)
    print("ERA5 Pressure Levels (India) — COMPLETE")
    print("=" * 60)
    print(f"Total wall time: {total_min:.1f} min")
    any_fails = sum(fail_summary.values())
    if any_fails:
        print(f"\n⚠ Failures by variable: {fail_summary}")
        print("  Re-run this cell — already-valid files will be skipped.")
    else:
        print("\n✓ No failures.")


Today is        : 2026-07-04
Year range      : 1980  →  2026
Requests needed : 93 half-year requests (was ~557 monthly requests before)
Parallel workers: 2
Output dir      : downloads_era5_pressure
Setup done.

u_wind_pl  (reanalysis-era5-pressure-levels)
  Scanning existing files... 34 valid, 59 missing


2c47ff244cdc7c8a29f9b0e51a19d342.nc:   0%|          | 0.00/122M [00:00<?, ?B/s]

    [  1/59] 1997-H1: done


d8386cfbb61a2277ff27131e98d84879.nc:   0%|          | 0.00/131M [00:00<?, ?B/s]

    [  2/59] 1997-H2: done


...

    [ 57/59] 2025-H1: done


543aca941d0d4f4c723280f59d841fbf.nc:   0%|          | 0.00/130M [00:00<?, ?B/s]

    [ 58/59] 2025-H2: done


493f703348c4ef3acf6867123d259025.nc:   0%|          | 0.00/95.8M [00:00<?, ?B/s]

    [ 59/59] 2026-H1: done
  → u_wind_pl: 59 downloaded, 34 pre-existing valid  (824.0 min)

ERA5 Pressure Levels (India) — COMPLETE
Total wall time: 824.0 min

✓ No failures.


In [ ]:
# ============================================================
# ERA5 Pressure Levels (0.25°) — parallel companion download
# CHANGE 1 APPLIED: batched by HALF-YEAR instead of by MONTH
# Same dataset (reanalysis-era5-pressure-levels), same
# resume/validity logic. A full year of hourly data across 3
# pressure levels for all of India hits CDS's "request too
# large" limit, so this batches by half-year (Jan-Jun, Jul-Dec)
# instead — still ~2 requests/year instead of 12.
# ============================================================

import os
import time
import warnings
import threading
from datetime import date
from concurrent.futures import ThreadPoolExecutor, as_completed

import cdsapi
import xarray as xr

warnings.filterwarnings("ignore", category=xr.SerializationWarning)

# ------------------------------------------------------------
# Setup
# ------------------------------------------------------------

DOWNLOAD_DIR = r"downloads_era5_pressure"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

# India Coordinates (CDS format: [North, West, South, East])
AREA = [37.1, 68.12, 6.75, 97.42]

ALL_MONTHS = [f"{m:02d}" for m in range(1, 13)]
DAYS = [f"{d:02d}" for d in range(1, 32)]
TIMES = [f"{h:02d}:00" for h in range(24)]

# Parallel downloads (recommended to keep low for CDS API)
N_WORKERS = 2

# ------------------------------------------------------------
# Dynamic date range
# ------------------------------------------------------------
# Download everything from Jan 1980 until two months before today
# (latest months are skipped because ERA5 production may lag)

START_YEAR, START_MONTH = 1980, 1
LAG_MONTHS = 2

today = date.today()
end_y, end_m = today.year, today.month

for _ in range(LAG_MONTHS):
    end_m -= 1
    if end_m == 0:
        end_m = 12
        end_y -= 1

# ------------------------------------------------------------
# Build list of available months for every year
# ------------------------------------------------------------

year_plan = []

for y in range(START_YEAR, end_y + 1):

    if y == START_YEAR and y == end_y:
        months = [f"{m:02d}" for m in range(START_MONTH, end_m + 1)]

    elif y == START_YEAR:
        months = [f"{m:02d}" for m in range(START_MONTH, 13)]

    elif y == end_y:
        months = [f"{m:02d}" for m in range(1, end_m + 1)]

    else:
        months = ALL_MONTHS

    year_plan.append((str(y), months))

# ------------------------------------------------------------
# Split every year into Jan-Jun (H1) and Jul-Dec (H2)
# ------------------------------------------------------------

H1_MONTHS = {f"{m:02d}" for m in range(1, 7)}
H2_MONTHS = {f"{m:02d}" for m in range(7, 13)}

period_plan = []

for year, months in year_plan:

    h1 = [m for m in months if m in H1_MONTHS]
    h2 = [m for m in months if m in H2_MONTHS]

    if h1:
        period_plan.append((year, "H1", h1))

    if h2:
        period_plan.append((year, "H2", h2))

print(f"Today is        : {today}")
print(f"Year range      : {year_plan[0][0]}  →  {year_plan[-1][0]}")
print(f"Requests needed : {len(period_plan)} half-year requests "
      f"(was ~{sum(len(m) for _, m in year_plan)} monthly requests before)")
print(f"Parallel workers: {N_WORKERS}")
print(f"Output dir      : {DOWNLOAD_DIR}")
print("Setup done.\n")

# ------------------------------------------------------------
# Variable specification
# ------------------------------------------------------------

VARIABLES = [
    (
        "u_wind_pl",
        "reanalysis-era5-pressure-levels",
        {
            "product_type": ["reanalysis"],
            "variable": ["u_component_of_wind"],
            "pressure_level": ["250"],
            "data_format": "netcdf",
            "download_format": "unarchived",
        },
    ),
]

# ============================================================
# Helper Functions
# ============================================================

def is_valid_nc(fpath):
    """
    Verify that a downloaded NetCDF file is usable.

    Opens the file using xarray and checks:
      - the file can be opened successfully
      - at least one data variable exists
      - the variable contains data

    Returns
    -------
    True  -> file is valid
    False -> corrupted, empty, or unreadable
    """
    try:
        with xr.open_dataset(fpath, engine="netcdf4") as d:

            dvs = list(d.data_vars)

            if not dvs:
                return False

            if d[dvs[0]].size == 0:
                return False

        return True

    except Exception:
        return False


def scan_existing(name, var_dir):
    """
    Scan the download directory before starting downloads.

    This function enables resume capability by:

      1. Removing leftover partial (.part) files.
      2. Checking every expected half-year file.
      3. Keeping valid files.
      4. Deleting corrupted files.
      5. Returning only the periods that still need downloading.

    Returns
    -------
    needs        : list of (year, half, months) still required
    n_valid      : number of valid files already present
    n_missing    : files that don't exist yet
    n_corrupt    : corrupted files removed
    n_partial    : stale partial downloads removed
    """

    needs = []

    n_valid = 0
    n_missing = 0
    n_corrupt = 0
    n_partial = 0

    # Remove interrupted download fragments
    for f in (os.listdir(var_dir) if os.path.isdir(var_dir) else []):

        if f.endswith(".part") or ".part." in f:

            try:
                os.remove(os.path.join(var_dir, f))
                n_partial += 1

            except OSError:
                pass

    # Check every required half-year file
    for year, half, months in period_plan:

        outfile = os.path.join(var_dir, f"{name}_{year}_{half}.nc")

        if not os.path.exists(outfile):

            needs.append((year, half, months))
            n_missing += 1

        elif is_valid_nc(outfile):

            n_valid += 1

        else:

            try:
                os.remove(outfile)
            except OSError:
                pass

            needs.append((year, half, months))
            n_corrupt += 1

    return needs, n_valid, n_missing, n_corrupt, n_partial


# Lock so print statements from multiple threads never overlap
_print_lock = threading.Lock()


def safe_print(msg):
    """
    Thread-safe printing.

    Multiple download threads may finish simultaneously.
    This lock ensures only one thread prints at a time so
    progress messages remain readable.
    """
    with _print_lock:
        print(msg, flush=True)


def download_one_period(name, dataset, extras, year, half, months, var_dir):
    """
    Download one half-year of ERA5 data.

    Steps
    -----
    1. Build the CDS API request.
    2. Download into a temporary '.part' file.
    3. Rename to the final filename only after success.
    4. Delete temporary files if anything fails.

    Returns
    -------
    (year, half, success, error_message)
    """

    final_path = os.path.join(var_dir, f"{name}_{year}_{half}.nc")
    part_path = f"{final_path}.part.{threading.get_ident()}"

    request = {
        **extras,
        "year": [year],
        "month": months,
        "day": DAYS,
        "time": TIMES,
        "area": AREA,
    }

    try:

        client = cdsapi.Client(
            quiet=True,
            wait_until_complete=True
        )

        client.retrieve(dataset, request, part_path)

        os.replace(part_path, final_path)

        return (year, half, True, None)

    except Exception as e:

        if os.path.exists(part_path):

            try:
                os.remove(part_path)
            except OSError:
                pass

        return (year, half, False, str(e))


def download_variable(name, dataset, extras):
    """
    Download an entire ERA5 variable.

    Workflow
    --------
    1. Scan existing files.
    2. Skip already-valid downloads.
    3. Launch parallel download workers.
    4. Monitor completion.
    5. Print a final summary.

    Returns
    -------
    Number of failed downloads.
    """

    var_dir = os.path.join(DOWNLOAD_DIR, name)
    os.makedirs(var_dir, exist_ok=True)

    print("=" * 60)
    print(f"{name}  ({dataset})")
    print("=" * 60)

    t0 = time.time()

    print("  Scanning existing files...", end=" ", flush=True)

    needs, n_valid, n_missing, n_corrupt, n_partial = scan_existing(
        name,
        var_dir,
    )

    parts = [f"{n_valid} valid"]

    if n_missing:
        parts.append(f"{n_missing} missing")

    if n_corrupt:
        parts.append(f"{n_corrupt} corrupted (deleted)")

    if n_partial:
        parts.append(f"{n_partial} partial (deleted)")

    print(", ".join(parts))

    if not needs:

        elapsed = (time.time() - t0) / 60.0
        print(f"  → nothing to do  ({elapsed:.1f} min)\n")

        return 0

    print(
        f"  Downloading {len(needs)} half-year period(s) "
        f"with {N_WORKERS} parallel workers..."
    )

    n_done = 0
    n_fail = 0

    completed = 0
    total = len(needs)

    with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:

        futures = {
            pool.submit(
                download_one_period,
                name,
                dataset,
                extras,
                year,
                half,
                months,
                var_dir,
            ): (year, half)

            for year, half, months in needs
        }

        for fut in as_completed(futures):

            year, half, ok, err = fut.result()

            completed += 1

            label = f"{year}-{half}"

            if ok:

                n_done += 1

                safe_print(
                    f"    [{completed:>3}/{total}] {label}: done"
                )

            else:

                n_fail += 1

                err_short = (
                    err[:200] + "..."
                    if len(err) > 200
                    else err
                )

                safe_print(
                    f"    [{completed:>3}/{total}] "
                    f"{label}: FAILED: {err_short}"
                )

                safe_print(
                    f"      → if this is still a volume/field-limit "
                    f"error, split {label}"
                )

                safe_print(
                    f"        into individual months and retry."
                )

    elapsed = (time.time() - t0) / 60.0

    summary = (
        f"  → {name}: {n_done} downloaded, "
        f"{n_valid} pre-existing valid"
    )

    if n_corrupt:
        summary += f", {n_corrupt} corrupted re-attempted"

    if n_fail:
        summary += f", {n_fail} FAILED"

    summary += f"  ({elapsed:.1f} min)"

    print(summary + "\n")

    return n_fail


# ============================================================
# Main
# ============================================================

if __name__ == "__main__":

    overall_t0 = time.time()

    fail_summary = {}

    for name, dataset, extras in VARIABLES:

        fails = download_variable(name, dataset, extras)

        fail_summary[name] = fails

    total_min = (time.time() - overall_t0) / 60.0

    print("=" * 60)
    print("ERA5 Pressure Levels (India) — COMPLETE")
    print("=" * 60)
    print(f"Total wall time: {total_min:.1f} min")

    any_fails = sum(fail_summary.values())

    if any_fails:

        print(f"\n⚠ Failures by variable: {fail_summary}")
        print("  Re-run this cell — already-valid files will be skipped.")

    else:

        print("\n✓ No failures.")
        

In [1]:
from dask.diagnostics import ProgressBar
#arcos script

# ============================================================
# ERA5 → DAILY (mean / max / min) for India, from ARCO-ERA5
#
# Why this design:
#   You're unsure whether you need max/min or just mean. Instead
#   of committing to one, this streams HOURLY ERA5 from Google's
#   ARCO store (no CDS queue) and computes daily mean, max, AND
#   min in one pass — so you keep every statistic. The hourly data
#   never lands on disk; only the small daily files are written.
#   Precipitation is accumulated, so it gets a daily SUM.
#
# Data: gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3
#       (the GraphCast/NeuralGCM superset; surface + 37 pressure levels)
#
# Deps: pip install xarray zarr gcsfs dask netCDF4 numpy
#
# NOTE: run over a couple of test years first (set END_YEAR small)
# to confirm variable names and volumes before the full pull.
# ============================================================

import os
import numpy as np
import xarray as xr

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------

ARCO_PATH = "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"
OUT_DIR = "era5_daily_india"
os.makedirs(OUT_DIR, exist_ok=True)

# India box. ARCO latitude runs 90 → -90 (descending), so the
# latitude slice is (North, South). Longitude is 0 → 359.75; India
# stays positive so no 0/360 wrap needed.
LAT_N, LAT_S = 37.1, 6.75
LON_W, LON_E = 68.12, 97.42

LEVELS = [250, 500, 850]

START_YEAR = 1980
END_YEAR = 2024          # start small (e.g. 1981) to test, then widen

# ---- Variable groups (ARCO analysis-ready names) ----
# Pressure-level, instantaneous → daily mean/max/min at LEVELS
PRESSURE_VARS = [
    "u_component_of_wind",
    "v_component_of_wind",
    "temperature",
    "specific_humidity",
]
# Relative humidity may not exist in the AR store. If missing it is
# derived from specific_humidity + temperature + pressure (see below).
PRESSURE_VARS_OPTIONAL = ["relative_humidity"]

# Surface, instantaneous → daily mean/max/min
# (the "at surface" analogues of your u/v/temp/humidity, plus MSLP)
SURFACE_INSTANT_VARS = [
    "mean_sea_level_pressure",
    "2m_temperature",
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",
    "2m_dewpoint_temperature",     # surface humidity proxy; drop if unwanted
]
# Surface, accumulated → daily SUM
SURFACE_ACCUM_VARS = ["total_precipitation"]

INSTANT_STATS = {"mean": "mean", "max": "max", "min": "min"}

DERIVE_RH_IF_MISSING = True

# ------------------------------------------------------------
# Open store (lazy)
# ------------------------------------------------------------

ds = xr.open_zarr(ARCO_PATH, chunks={"time": 24 * 31},
                  storage_options={"token": "anon"})

# Clamp to the data actually available in the store
avail_start = np.datetime64(ds.attrs["valid_time_start"])
avail_stop = np.datetime64(ds.attrs["valid_time_stop"])

present = set(ds.data_vars)

def keep(names):
    ok, missing = [v for v in names if v in present], [v for v in names if v not in present]
    for m in missing:
        print(f"  ⚠ not in store, skipping: {m}")
    return ok

pl_vars = keep(PRESSURE_VARS)
pl_opt = keep(PRESSURE_VARS_OPTIONAL)
sfc_inst = keep(SURFACE_INSTANT_VARS)
sfc_acc = keep(SURFACE_ACCUM_VARS)

need_rh = ("relative_humidity" in PRESSURE_VARS_OPTIONAL
           and "relative_humidity" not in pl_opt
           and DERIVE_RH_IF_MISSING)

print(f"ARCO availability : {avail_start} → {avail_stop}")
print(f"Pressure vars     : {pl_vars + pl_opt}"
      f"{'  (+ derived relative_humidity)' if need_rh else ''}")
print(f"Surface instant   : {sfc_inst}")
print(f"Surface accum(sum): {sfc_acc}")
print(f"Levels            : {LEVELS}")
print(f"Years             : {START_YEAR}–{END_YEAR}\n")

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def derive_rh(temp_K, q, level_hPa):
    """Relative humidity (%) from specific humidity, temperature, pressure.
    Bolton (1980) saturation vapour pressure. level_hPa broadcasts over 'level'."""
    e = q * level_hPa / (0.622 + 0.378 * q)                 # vapour pressure (hPa)
    es = 6.112 * np.exp(17.67 * (temp_K - 273.15) / (temp_K - 29.65))
    return (100.0 * e / es).clip(0, 100)


def is_valid(path):
    try:
        with xr.open_dataset(path) as d:
            return len(d.data_vars) > 0 and d[list(d.data_vars)[0]].size > 0
    except Exception:
        return False


def daily_stats_for_month(month_ds, accum_vars):
    """Return a dataset of daily stats for one month, already in memory."""
    g = month_ds.resample(time="1D")
    out = {}
    for v in month_ds.data_vars:
        if v in accum_vars:
            out[f"{v}_sum"] = g.sum()[v]
        else:
            for suffix, how in INSTANT_STATS.items():
                out[f"{v}_{suffix}"] = getattr(g, how)()[v]
    return xr.Dataset(out)


def process_year(year):
    out_path = os.path.join(OUT_DIR, f"era5_daily_india_{year}.nc")
    if os.path.exists(out_path):
        if is_valid(out_path):
            print(f"  {year}: already done, skipping")
            return
        os.remove(out_path)

    y0 = np.datetime64(f"{year}-01-01T00")
    y1 = np.datetime64(f"{year}-12-31T23")
    if y1 < avail_start or y0 > avail_stop:
        print(f"  {year}: outside ARCO range, skipping")
        return

    monthly = []
    for m in range(1, 13):
        t0 = np.datetime64(f"{year}-{m:02d}-01T00")
        t1 = (np.datetime64(f"{year}-{m+1:02d}-01T00") - np.timedelta64(1, "h")
              if m < 12 else np.datetime64(f"{year}-12-31T23"))
        if t1 < avail_start or t0 > avail_stop:
            continue

        # Pressure-level view (subset levels), surface view (no level dim)
        pl = ds[pl_vars + pl_opt].sel(
            level=LEVELS,
            latitude=slice(LAT_N, LAT_S),
            longitude=slice(LON_W, LON_E),
            time=slice(t0, t1),
        )
        sfc = ds[sfc_inst + sfc_acc].sel(
            latitude=slice(LAT_N, LAT_S),
            longitude=slice(LON_W, LON_E),
            time=slice(t0, t1),
        )

        # Pull this month into memory once (bounded ~<1 GB for India box)
        print(f"    {year}-{m:02d} fetching from Google Cloud...")
        with ProgressBar():
            pl = pl.load()
            sfc = sfc.load()

        if need_rh and "specific_humidity" in pl and "temperature" in pl:
            pl["relative_humidity"] = derive_rh(
                pl["temperature"], pl["specific_humidity"], pl["level"]
            )

        pl_daily = daily_stats_for_month(pl, accum_vars=[])
        sfc_daily = daily_stats_for_month(sfc, accum_vars=sfc_acc)
        monthly.append(xr.merge([pl_daily, sfc_daily]))
        print(f"    {year}-{m:02d} done")

    if not monthly:
        print(f"  {year}: no data in range")
        return

    year_ds = xr.concat(monthly, dim="time")
    comp = {v: {"zlib": True, "complevel": 4} for v in year_ds.data_vars}
    year_ds.to_netcdf(out_path, encoding=comp)
    print(f"  {year}: written → {out_path}\n")


# ------------------------------------------------------------
# Main
# ------------------------------------------------------------

if __name__ == "__main__":
    for year in range(START_YEAR, END_YEAR + 1):
        try:
            process_year(year)
        except Exception as e:
            print(f"  {year}: FAILED — {e}\n")
    print("Done. Re-run to fill any years that failed (existing years are skipped).")

I0705 18:59:49.088963  136243 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
I0705 18:59:49.096986  136460 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(82, generation: 1)


  ⚠ not in store, skipping: relative_humidity
ARCO availability : 1940-01-01 → 2025-12-31
Pressure vars     : ['u_component_of_wind', 'v_component_of_wind', 'temperature', 'specific_humidity']  (+ derived relative_humidity)
Surface instant   : ['mean_sea_level_pressure', '2m_temperature', '10m_u_component_of_wind', '10m_v_component_of_wind', '2m_dewpoint_temperature']
Surface accum(sum): ['total_precipitation']
Levels            : [250, 500, 850]
Years             : 1980–2024

    1980-01 fetching from Google Cloud...
[                                        ] | 0% Completed | 833.30 ms


KeyboardInterrupt: 